# Wav2Vec2 Arabic -> OpenVINO, mit Live-Backend für die Mobile-App

Zwei Teile in einem Notebook:

**Teil A (Schritte 1-9): Export.** Konvertiert
**`jonatasgrosman/wav2vec2-large-xlsr-53-arabic`** nach OpenVINO IR (FP32 + INT8)
für CPU-Inferenz und prüft die **Logit-Treue** — nicht nur das Transkript.

**Teil B (Schritte 10-14): Live-Backend.** Startet FastAPI + Tailscale-Funnel mit
dem OpenVINO-Modell, sodass die Mobile-App direkt dagegen testen kann. Scoring,
Preprocessing, FastAPI und Tailscale sind **wörtlich** aus
`wav2vec2_arabic_pronunciation.ipynb` übernommen; ausgetauscht wird allein die
Modell-Ladezelle durch einen Drop-in-Shim (Schritt 11).

**Wichtig zum Format:** `OVModelForCTC(export=True)` exportiert PyTorch → OpenVINO IR
(`.xml`/`.bin`) **direkt** über `torch.export`/TorchScript, *nicht* über ONNX.
ONNX als Zwischenformat ist unnötig und kostet nur Genauigkeit und Zeit.
Ein separater ONNX-Export steht als optionaler Schritt 8 bereit.

**Warum die Logit-Treue-Prüfung (Schritt 5):** Das Backend bewertet nicht per WER,
sondern liest `log_softmax` direkt für GOP-/LLR-Scoring aus. WER ist robust gegen
Logit-Rauschen, GOP-Scores sind es **nicht**. Ein identisches Transkript ist
deshalb *kein* ausreichender Abnahmetest für INT8.

| Schritt | Inhalt | nötig fürs App-Testen |
|---|---|---|
| 1 | Installation (+ Runtime-Restart) | **ja** |
| 2 | Versionen prüfen, PyTorch-Referenz laden | **ja** |
| 3 | Export nach OpenVINO IR | **ja** |
| 4 | Testaudio + erste Inferenz | empfohlen |
| 5 | Logit-Treue FP32 vs. PyTorch | empfohlen |
| 6 | CPU-Tuning + Benchmark | optional |
| 7 | INT8-Quantisierung mit NNCF | nur für `VARIANT="int8"` |
| 8 | Optional: separater ONNX-Export | nein |
| 9 | Artefakte packen + Deployment-Snippet | nein |
| 10 | Imports für das Backend | **ja** |
| 11 | **OpenVINO-Modell laden (Shim)** | **ja** |
| 12 | Audio-Preprocessing + Scoring | **ja** |
| 13 | FastAPI-App (`/assess`, `/stream`, `/health`, `/logs`) | **ja** |
| 14 | Tailscale-Funnel + Server starten → **URL für die App** | **ja** |

**Runtime-Wahl:** CPU-Runtime nehmen — das ist der Testzweck. Auf einer
GPU-Runtime läuft der ASR-Forward trotzdem auf CPU (Schritt 11 erzwingt das),
aber die Messwerte wären nicht vergleichbar.

## Schritt 1 — Installation

Nach dieser Zelle **startet die Runtime automatisch neu** (`os.kill`). Das ist
beabsichtigt: `openvino`/`nncf` binden `numpy` neu, und ohne Restart crasht
`librosa`/`numba`. Danach **ab Schritt 2 weiterlaufen lassen** — Zelle 1 nicht
erneut ausführen.

In [ ]:
# Installation: OpenVINO-Export + Live-Backend fuer die Mobile-App.
# Danach startet die Runtime neu -> weiter bei Schritt 2.
!pip install -q "optimum-intel[openvino]" nncf transformers soundfile librosa gTTS datasets
!pip install -q -U pydub nest_asyncio python-multipart fastapi 'uvicorn[standard]' silero-vad
!apt-get -qq install -y ffmpeg > /dev/null

# scipy bewusst NICHT upgraden: Colab liefert es vorinstalliert, ein Upgrade in
# laufender Session gibt einen Cython-ABI-Mismatch (scipy._cyutility ImportError).

# Falls Imports in Schritt 2 fehlschlagen, stattdessen die "eager"-Variante:
# !pip install -q -U --upgrade-strategy eager "optimum[openvino,nncf]" transformers soundfile librosa gTTS datasets

import os
os.kill(os.getpid(), 9)   # Runtime-Restart: openvino/nncf binden numpy neu

## Schritt 2 — Versionen prüfen, PyTorch-Referenz laden

Die PyTorch-FP32-Ausgabe ist die **Referenz**, gegen die später FP32-IR und INT8-IR
gemessen werden. `blank_id` (= `config.pad_token_id`) und das Frame-Raster (~20 ms)
sind identisch zum Backend.

In [ ]:
import numpy as np, torch, transformers, openvino as ov, nncf, optimum.intel

print("transformers  ", transformers.__version__)
print("torch         ", torch.__version__)
print("openvino      ", ov.__version__)
print("nncf          ", nncf.__version__)
print("optimum-intel ", optimum.intel.__version__)
print("CPU-Device    ", ov.Core().get_property("CPU", "FULL_DEVICE_NAME"))

from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

MODEL_ID    = "jonatasgrosman/wav2vec2-large-xlsr-53-arabic"
OV_FP32_DIR = "ov_wav2vec2_ar_fp32"
OV_INT8_DIR = "ov_wav2vec2_ar_int8"

processor   = Wav2Vec2Processor.from_pretrained(MODEL_ID)
torch_model = Wav2Vec2ForCTC.from_pretrained(MODEL_ID).eval()

SR       = processor.feature_extractor.sampling_rate      # 16000
BLANK_ID = torch_model.config.pad_token_id                # CTC-Blank
VOCAB    = torch_model.config.vocab_size

print(f"\nSampleRate={SR}  vocab={VOCAB}  blank_id={BLANK_ID}")
print(f"Params={sum(p.numel() for p in torch_model.parameters())/1e6:.1f}M")

## Schritt 3 — Export nach OpenVINO IR

`compile=False`: erst speichern, kompiliert wird später mit explizitem CPU-Config
(Schritt 6). Der `processor` **muss** mitgespeichert werden — ohne `vocab.json` /
`preprocessor_config.json` ist im Deployment kein Decoding möglich.

In [ ]:
from optimum.intel import OVModelForCTC

ov_model = OVModelForCTC.from_pretrained(MODEL_ID, export=True, compile=False)

ov_model.save_pretrained(OV_FP32_DIR)
processor.save_pretrained(OV_FP32_DIR)     # vocab/tokenizer/feature-extractor mitspeichern

!ls -la {OV_FP32_DIR}
print("\nIR-Groesse:")
!du -sh {OV_FP32_DIR}

# Shape-Signatur des IR pruefen (dynamische Sequenzlaenge = [1,?])
m = ov.Core().read_model(f"{OV_FP32_DIR}/openvino_model.xml")
for i in m.inputs:  print("IN :", i.get_any_name(), i.partial_shape)
for o in m.outputs: print("OUT:", o.get_any_name(), o.partial_shape)

## Schritt 4 — Testaudio + erste Inferenz

Das Testaudio wird per gTTS synthetisiert, damit die Zelle ohne Datensatz-Auth
läuft. Für belastbare Messungen eigene Aufnahmen nutzen (Upload-Block unten) —
synthetische Stimmen sind für ASR-Bewertung nicht repräsentativ.

In [ ]:
import librosa
from gtts import gTTS

TEXT = "بسم الله الرحمن الرحيم"
gTTS(TEXT, lang="ar").save("sample_ar.mp3")
audio, _ = librosa.load("sample_ar.mp3", sr=SR, mono=True)   # bei mp3-Fehler: !ffmpeg -i sample_ar.mp3 sample_ar.wav
print(f"Audio: {len(audio)} samples = {len(audio)/SR:.2f}s")

# --- Optional: eigene Datei hochladen -----------------------------
# from google.colab import files
# up = files.upload()
# audio, _ = librosa.load(next(iter(up)), sr=SR, mono=True)

def features(a):
    """Genau die Vorverarbeitung, die auch asr_app.py nutzt."""
    return processor(a, sampling_rate=SR, return_tensors="np",
                     padding=False).input_values.astype(np.float32)

def greedy(logits):
    ids = np.asarray(logits).argmax(-1)
    return processor.batch_decode(torch.as_tensor(ids))[0]

feats = features(audio)

# PyTorch-Referenz (FP32)
with torch.no_grad():
    logits_pt = torch_model(torch.from_numpy(feats)).logits.numpy()

# OpenVINO IR (FP32)
ov_fp32   = OVModelForCTC.from_pretrained(OV_FP32_DIR)
logits_ov = np.asarray(ov_fp32(torch.from_numpy(feats)).logits)

print("logits shape :", logits_pt.shape, "->", logits_pt.shape[1], "Frames a ~20ms")
print("PyTorch      :", greedy(logits_pt))
print("OpenVINO     :", greedy(logits_ov))

## Schritt 5 — Logit-Treue prüfen

Abnahmekriterien statt Transkript-Vergleich:

* **FP32-IR:** `max|dLogit| < 1e-3`, Top-1-Frame-Match `100 %`, `dGOP` praktisch `0`.
* **INT8 (Schritt 7):** entscheidend sind `mean|dLogProb|` und `dGOP`. Driftet `dGOP`
  um mehr als ~`0.05`, sind die Buchstaben-Schwellen im Backend nicht mehr
  kalibriert -> dann FP32 behalten oder `ignored_scope` aktivieren.

In [ ]:
def log_softmax(x, axis=-1):
    m = x.max(axis=axis, keepdims=True)
    e = np.exp(x - m)
    return x - m - np.log(e.sum(axis=axis, keepdims=True))

def compare(ref, cand, name):
    assert ref.shape == cand.shape, f"Frame-Mismatch {ref.shape} vs {cand.shape}"
    lp_r, lp_c = log_softmax(ref), log_softmax(cand)
    d_logit    = np.abs(ref - cand)
    d_logprob  = np.abs(lp_r - lp_c)
    top1       = (ref.argmax(-1) == cand.argmax(-1)).mean()
    corr       = np.corrcoef(ref.ravel(), cand.ravel())[0, 1]
    # Effekt auf den tatsaechlich gescorten Wert: logprob des Greedy-Pfads
    idx   = ref.argmax(-1)[0]
    gop_r = lp_r[0, np.arange(len(idx)), idx].mean()
    gop_c = lp_c[0, np.arange(len(idx)), idx].mean()
    print(f"--- {name} ---")
    print(f"  max |dLogit|        : {d_logit.max():.4f}")
    print(f"  mean|dLogProb|      : {d_logprob.mean():.5f}   max: {d_logprob.max():.4f}")
    print(f"  Top-1-Frame-Match   : {top1*100:.2f} %")
    print(f"  Pearson r           : {corr:.6f}")
    print(f"  mean GOP (ref/cand) : {gop_r:.4f} / {gop_c:.4f}  -> d={abs(gop_r-gop_c):.4f}")
    print(f"  Transkript identisch: {greedy(ref) == greedy(cand)}")

compare(logits_pt, logits_ov, "OpenVINO FP32 vs PyTorch FP32")

## Schritt 6 — CPU-Tuning und Benchmark

Drei Varianten im Vergleich: PyTorch FP32, OpenVINO FP32 mit dynamischer Shape,
OpenVINO FP32 mit **statischer** Shape.

* `PERFORMANCE_HINT=LATENCY` optimiert die Einzelanfrage, nicht den Durchsatz.
* `CACHE_DIR` legt das kompilierte Blob ab -> kürzerer Cold-Start (Verzeichnis mit
  ins Container-Image legen).
* **Dynamische Shapes** lassen OpenVINO bei jeder neuen Länge intern
  re-optimieren -> die erste Anfrage pro Länge ist langsamer. Für ein
  Latenzbudget < 300 ms lohnt **Bucketing**: auf 2 s / 4 s / 8 s padden und je
  Bucket ein kompiliertes Modell halten.
* Colab-CPU (2 geteilte vCPU, kein garantiertes AVX-512/AMX) ist **nicht**
  repräsentativ — die Zahlen hier sind Relativwerte. Absolut auf der Zielhardware
  messen.

In [ ]:
import time, os

N_THREADS = os.cpu_count()
OV_CONFIG = {
    "PERFORMANCE_HINT":      "LATENCY",       # Einzelanfrage schnell, nicht Throughput
    "NUM_STREAMS":           "1",
    "INFERENCE_NUM_THREADS": str(N_THREADS),  # im Container = zugeteilte Kerne
    "CACHE_DIR":             "ov_cache",      # kompiliertes Blob cachen -> Cold-Start
}

def bench(fn, x, warmup=3, runs=15):
    for _ in range(warmup): fn(x)
    ts = []
    for _ in range(runs):
        t0 = time.perf_counter(); fn(x); ts.append((time.perf_counter() - t0) * 1e3)
    ts = np.array(ts)
    return ts.mean(), np.median(ts), ts.min()

dur = len(audio) / SR
xt  = torch.from_numpy(feats)

# a) PyTorch FP32
torch.set_num_threads(N_THREADS)
with torch.no_grad():
    r_pt = bench(lambda x: torch_model(x).logits, xt)

# b) OpenVINO FP32, dynamische Shape
ov_dyn = OVModelForCTC.from_pretrained(OV_FP32_DIR, ov_config=OV_CONFIG)
r_dyn  = bench(lambda x: ov_dyn(x), xt)

# c) OpenVINO FP32, STATISCHE Shape (auf 4s gepaddet)
FIXED_LEN = SR * 4
def pad_fixed(a, n=FIXED_LEN):
    a = a[:n]
    return np.pad(a, (0, n - len(a)))

ov_st = OVModelForCTC.from_pretrained(OV_FP32_DIR, ov_config=OV_CONFIG, compile=False)
ov_st.reshape(1, FIXED_LEN)      # Batch 1, feste Sequenzlaenge
ov_st.compile()
xs   = torch.from_numpy(features(pad_fixed(audio)))
r_st = bench(lambda x: ov_st(x), xs)

print(f"Audio {dur:.2f}s | {N_THREADS} Threads     mean/median/min ms      RTF(med)")
for name, r, d in [("PyTorch FP32",       r_pt,  dur),
                   ("OpenVINO FP32 dyn",  r_dyn, dur),
                   ("OpenVINO FP32 stat", r_st,  4.0)]:
    print(f"  {name:<20} {r[0]:7.1f} {r[1]:7.1f} {r[2]:7.1f}      {r[1]/1e3/d:.4f}")

## Schritt 7 — INT8-Quantisierung mit NNCF

Bewusst `nncf.quantize` direkt auf dem IR statt `OVQuantizer`: API-stabiler und
volle Kontrolle über `model_type` und `ignored_scope`.

* `model_type=ModelType.TRANSFORMER` schützt LayerNorm-/Softmax-/Attention-Pfade.
* **Kalibrierdaten sind der Qualitätsfaktor.** Priorität: (1) eigene Aufnahmen in
  `/content/calib_audio/`, (2) FLEURS `ar_eg`, (3) gTTS — letzteres validiert nur
  die Pipeline, nicht die Genauigkeit.
* Full-INT8 (nicht Weight-only) ist hier richtig: wav2vec2 ist compute-bound.
  Weight-only-Kompression hilft vor allem memory-bound LLM-Decoding.
* Driften die GOP-Werte zu stark, den Conv-Feature-Extractor per `ignored_scope`
  in FP32 belassen — er ist der empfindlichste Teil von wav2vec2.

In [ ]:
import glob, shutil, nncf

# --- Kalibrierdaten: 1) eigene Aufnahmen, 2) FLEURS ar_eg, 3) gTTS ---
N_CALIB = 64
calib_audios = []

local = sorted(glob.glob("/content/calib_audio/*.wav") + glob.glob("/content/calib_audio/*.mp3"))
if local:
    calib_audios = [librosa.load(p, sr=SR, mono=True)[0] for p in local[:N_CALIB]]
    print(f"Kalibrierung: {len(calib_audios)} eigene Dateien (bester Fall)")
else:
    try:
        from datasets import load_dataset
        ds = load_dataset("google/fleurs", "ar_eg", split="validation", streaming=True)
        for i, s in enumerate(ds):
            if i >= N_CALIB: break
            a = np.asarray(s["audio"]["array"], dtype=np.float32)
            if s["audio"]["sampling_rate"] != SR:
                a = librosa.resample(a, orig_sr=s["audio"]["sampling_rate"], target_sr=SR)
            calib_audios.append(a[:SR * 8])
        print(f"Kalibrierung: {len(calib_audios)} FLEURS ar_eg Samples")
    except Exception as e:
        print("FLEURS nicht verfuegbar:", e, "-> Fallback gTTS")
        phrases = ["بسم الله الرحمن الرحيم", "الحمد لله رب العالمين",
                   "الرحمن الرحيم", "مالك يوم الدين", "إياك نعبد وإياك نستعين",
                   "اهدنا الصراط المستقيم", "قل هو الله أحد", "الله الصمد"]
        for i, p in enumerate(phrases):
            gTTS(p, lang="ar").save(f"c{i}.mp3")
            calib_audios.append(librosa.load(f"c{i}.mp3", sr=SR, mono=True)[0])
        print(f"Kalibrierung: {len(calib_audios)} gTTS-Samples "
              "(NUR Pipeline-Test - synthetische Stimmen sind eine schlechte "
              "Kalibrierverteilung fuer Kinderstimmen!)")

# --- Quantisieren ---------------------------------------------------
core       = ov.Core()
model_fp32 = core.read_model(f"{OV_FP32_DIR}/openvino_model.xml")
IN_NAME    = model_fp32.inputs[0].get_any_name()

calib_ds = nncf.Dataset(calib_audios, lambda a: {IN_NAME: features(a)})

model_int8 = nncf.quantize(
    model_fp32,
    calib_ds,
    model_type=nncf.ModelType.TRANSFORMER,   # schuetzt LayerNorm/Softmax/Attention
    subset_size=len(calib_audios),
    # Bei zu starkem GOP-Drift aktivieren:
    # ignored_scope=nncf.IgnoredScope(patterns=[".*feature_extractor.*"]),
)

os.makedirs(OV_INT8_DIR, exist_ok=True)
ov.save_model(model_int8, f"{OV_INT8_DIR}/openvino_model.xml")
for f in glob.glob(f"{OV_FP32_DIR}/*.json") + glob.glob(f"{OV_FP32_DIR}/*.txt"):
    shutil.copy(f, OV_INT8_DIR)             # config, preprocessor_config, vocab, tokenizer

!du -sh {OV_FP32_DIR} {OV_INT8_DIR}

# --- Qualitaet + Latenz gegen FP32 ----------------------------------
ov_int8   = OVModelForCTC.from_pretrained(OV_INT8_DIR, ov_config=OV_CONFIG)
logits_i8 = np.asarray(ov_int8(xt).logits)
compare(logits_pt, logits_i8, "OpenVINO INT8 vs PyTorch FP32")

r_i8 = bench(lambda x: ov_int8(x), xt)
print(f"\nINT8 dyn: mean {r_i8[0]:.1f} ms | median {r_i8[1]:.1f} ms "
      f"| Speedup vs OV-FP32 {r_dyn[1]/r_i8[1]:.2f}x | vs PyTorch {r_pt[1]/r_i8[1]:.2f}x")

## Schritt 8 — Optional: separater ONNX-Export

Nicht Teil des OpenVINO-Pfads (Schritt 3 geht direkt zu IR). Nur sinnvoll, wenn
gegen ONNX Runtime verglichen werden soll.

In [ ]:
!optimum-cli export onnx \
    --model {MODEL_ID} \
    --task automatic-speech-recognition \
    --opset 17 \
    onnx_wav2vec2_ar/
!ls -la onnx_wav2vec2_ar/

# IR liesse sich auch daraus bauen (aequivalent, nur Umweg):
# !ovc onnx_wav2vec2_ar/model.onnx --output_model ir_from_onnx/openvino_model.xml

## Schritt 9 — Artefakte packen und Deployment

Die zweite Zelle ist der Laufzeit-Code für den CPU-Container: **`openvino` +
`transformers` + `numpy`, kein `torch`, kein `optimum`** -> deutlich kleineres Image
und kürzerer Cold-Start. `transformers` bleibt nur für Feature-Extractor und
Tokenizer drin (beide reines NumPy).

Das Frame-Raster bleibt identisch (~20 ms/Frame, gleicher Conv-Stack), die
zurückgegebenen `logprobs` sind also direkt in die bestehende GOP-/LLR-Logik
einsetzbar.

In [ ]:
!zip -qr ov_wav2vec2_ar.zip {OV_FP32_DIR} {OV_INT8_DIR}
!du -sh ov_wav2vec2_ar.zip

from google.colab import files
files.download("ov_wav2vec2_ar.zip")

# Alternativ direkt auf den HF-Hub:
# from huggingface_hub import notebook_login; notebook_login()
# ov_int8.push_to_hub(OV_INT8_DIR, repository_id="<user>/wav2vec2-ar-ov-int8")

In [ ]:
# ---- Laufzeit-Code fuer den CPU-Container (ohne torch/optimum) ----
import numpy as np, openvino as ov, librosa
from transformers import Wav2Vec2Processor

MODEL_DIR = "ov_wav2vec2_ar_int8"

processor = Wav2Vec2Processor.from_pretrained(MODEL_DIR)
core      = ov.Core()
compiled  = core.compile_model(
    f"{MODEL_DIR}/openvino_model.xml", "CPU",
    {"PERFORMANCE_HINT": "LATENCY", "NUM_STREAMS": "1",
     "INFERENCE_NUM_THREADS": "4", "CACHE_DIR": "/root/ov_cache"},
)
req      = compiled.create_infer_request()     # wiederverwenden = weniger Overhead
IN, OUT  = compiled.input(0), compiled.output(0)
BLANK_ID = processor.tokenizer.pad_token_id

def log_softmax_np(x, axis=-1):
    m = x.max(axis=axis, keepdims=True)
    e = np.exp(x - m)
    return x - m - np.log(e.sum(axis=axis, keepdims=True))

def transcribe(wav_path_or_array):
    a = (librosa.load(wav_path_or_array, sr=16000, mono=True)[0]
         if isinstance(wav_path_or_array, str) else wav_path_or_array)
    x = processor(a, sampling_rate=16000, return_tensors="np").input_values.astype(np.float32)
    logits   = req.infer({IN: x})[OUT]            # (1, T, vocab), T = ~20ms/Frame
    logprobs = log_softmax_np(logits)             # -> direkt fuer GOP-/LLR-Scoring
    text     = processor.tokenizer.decode(logits[0].argmax(-1))
    return text, logprobs

text, lp = transcribe("sample_ar.mp3")
print("Transkript:", text)
print("logprobs  :", lp.shape, "| blank_id:", BLANK_ID)

---

# Teil B — Live-Backend für die Mobile-App

Ab hier läuft das Backend aus `wav2vec2_arabic_pronunciation.ipynb`, aber mit dem
OpenVINO-Modell aus Teil A. **Voraussetzung: Schritt 1-3 sind gelaufen** (und
Schritt 7, wenn `VARIANT = "int8"`), denn die IR-Verzeichnisse müssen auf der
Session-Disk liegen.

Wurde die Runtime zwischendurch neu gestartet, sind die Verzeichnisse noch da —
wurde die **VM** recycelt, Schritt 1-3 erneut ausführen.

## Schritt 10 — Speicher freigeben, dann Backend-Imports

Teil A hält je nach ausgeführten Schritten mehrere kompilierte Modelle im RAM
(PyTorch-Referenz ~1.2 GB, FP32-IR dynamisch, FP32-IR statisch, INT8-IR). Die
erste Zelle wirft alles weg, was Teil B nicht braucht — auf einer Colab-CPU-Runtime
sonst ein realistischer OOM-Kandidat.

Die zweite Zelle ist wörtlich aus `wav2vec2_arabic_pronunciation.ipynb` (Zelle 2).
`device` wird dort auf CUDA gesetzt, falls verfügbar — Schritt 11 überschreibt das
auf CPU.

In [ ]:
# Teil-A-Modelle freigeben, bevor Teil B sein eigenes kompiliert.
import gc
for _n in ("torch_model", "ov_model", "ov_fp32", "ov_dyn", "ov_st", "ov_int8",
           "model_fp32", "model_int8", "calib_ds", "calib_audios",
           "compiled", "req", "core"):
    globals().pop(_n, None)     # pop statt del: Schritte 6-9 sind optional
gc.collect()

try:
    import psutil, os as _os
    print(f"RSS jetzt: {psutil.Process(_os.getpid()).memory_info().rss/1e9:.2f} GB")
except Exception:
    pass

In [ ]:
import io, os, re, time, subprocess, threading, urllib.request, unicodedata
from typing import List, Dict, Any

import numpy as np
import torch
import torchaudio.functional as AF
from pydub import AudioSegment
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
from silero_vad import load_silero_vad, get_speech_timestamps

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SR = 16000
print(f"✅ Device: {device}, torch {torch.__version__}")

## Schritt 11 — OpenVINO-Modell laden (ersetzt die PyTorch-Ladezelle)

Das ist die **einzige** inhaltliche Änderung gegenüber dem laufenden Notebook.
`OVCTCModel` ist ein Drop-in-Ersatz für `Wav2Vec2ForCTC`: das übrige Notebook
braucht davon nur `__call__(input_values) -> .logits` und `.config.pad_token_id`.
Dadurch bleiben Preprocessing, Scoring und FastAPI unverändert — keine Divergenz
zum bereits funktionierenden Code.

* `VARIANT` schaltet zwischen `"int8"` (Schritt 7) und `"fp32"` (Schritt 3).
* Ein `threading.Lock` serialisiert die Inferenz: eine `InferRequest` ist nicht
  thread-safe, und der FastAPI-Pfad ruft über `asyncio.to_thread` auf. Das
  entspricht `max_inputs=1` im Modal-Deployment.
* Das Output-Array wird kopiert, **bevor** das Lock fällt — `infer()` gibt einen
  View auf den wiederverwendeten Output-Tensor der Request zurück.

In [ ]:
# ---- OpenVINO statt PyTorch: Drop-in-Ersatz fuer asr_model ----------------
import threading
from types import SimpleNamespace
import openvino as ov

VARIANT   = "int8"          # "int8" (Schritt 7) oder "fp32" (Schritt 3)
MODEL_DIR = OV_INT8_DIR if VARIANT == "int8" else OV_FP32_DIR
assert os.path.exists(f"{MODEL_DIR}/openvino_model.xml"), (
    f"{MODEL_DIR} fehlt - erst Schritt 3"
    + (" und Schritt 7" if VARIANT == "int8" else "") + " ausfuehren.")

ASR_MODEL_ID = f"jonatasgrosman/wav2vec2-large-xlsr-53-arabic (OpenVINO {VARIANT.upper()})"

# Felder, die /health und run_asr aus dem Original-Notebook erwarten.
USE_FP16 = False
DTYPE    = torch.float32
device   = torch.device("cpu")          # OpenVINO laeuft hier bewusst auf CPU

N_THREADS = os.cpu_count()
OV_RUNTIME_CONFIG = {
    "PERFORMANCE_HINT":      "LATENCY",
    "NUM_STREAMS":           "1",
    "INFERENCE_NUM_THREADS": str(N_THREADS),
    "CACHE_DIR":             "ov_cache",
}


class OVCTCModel:
    """Drop-in-Ersatz fuer Wav2Vec2ForCTC auf Basis eines OpenVINO-IR.

    Gebraucht wird vom restlichen Notebook nur __call__(input_values) -> .logits
    und .config.pad_token_id. Das Lock serialisiert die Inferenz (InferRequest
    ist nicht thread-safe, FastAPI ruft via asyncio.to_thread auf).
    """

    def __init__(self, model_dir: str, cfg: dict):
        core          = ov.Core()
        self.compiled = core.compile_model(f"{model_dir}/openvino_model.xml", "CPU", cfg)
        self.req      = self.compiled.create_infer_request()
        self._in      = self.compiled.input(0)
        self._out     = self.compiled.output(0)
        self._lock    = threading.Lock()
        self.config   = SimpleNamespace(pad_token_id=None)   # unten gesetzt

    def __call__(self, input_values):
        x = input_values.detach().cpu().numpy().astype(np.float32)
        with self._lock:
            # .copy() noch im Lock: infer() liefert einen View auf den
            # wiederverwendeten Output-Tensor der Request.
            logits = self.req.infer({self._in: x})[self._out].copy()
        return SimpleNamespace(logits=torch.from_numpy(logits))

    def eval(self):      # API-Kompatibilitaet, no-op
        return self


print(f"Lade OpenVINO-Modell aus {MODEL_DIR}  ({N_THREADS} Threads) …")
asr_processor = Wav2Vec2Processor.from_pretrained(MODEL_DIR)
asr_model     = OVCTCModel(MODEL_DIR, OV_RUNTIME_CONFIG)
ASR_VOCAB     = asr_processor.tokenizer.get_vocab()
ASR_BLANK_ID  = asr_processor.tokenizer.pad_token_id
asr_model.config.pad_token_id = ASR_BLANK_ID
print(f"  OK  {len(ASR_VOCAB)} tokens, blank={ASR_BLANK_ID}, variant={VARIANT}")

print("Lade Silero VAD …")
vad_model = load_silero_vad()
print("  OK  Silero VAD 5")

# Warm-up mit 2s Null-Audio. Bei dynamischer Shape optimiert OpenVINO pro neuer
# Laenge nach -> die erste Anfrage je Audiolaenge bleibt langsamer.
_t0 = time.perf_counter()
_ = asr_model(torch.zeros(1, 2 * SR)).logits
print(f"Modelle geladen und aufgewaermt ({(time.perf_counter()-_t0)*1e3:.0f} ms Warm-up-Forward).")

## Schritt 12 — Audio-Preprocessing und Scoring

Beide Zellen wörtlich aus `wav2vec2_arabic_pronunciation.ipynb` (Zellen 4 und 5).
Die Signalkette (`decode → HP 80 Hz → RMS-Norm → gentle_trim → Kontext-Pad`), das
GOP-/LLR-Scoring, `forced_align` und die Tajweed-Schwellen sind unverändert — nur
so sind die Ergebnisse mit dem GPU-Backend vergleichbar.

`run_asr` in der zweiten Zelle ruft `asr_model(...)` auf und trifft damit
automatisch den OpenVINO-Shim aus Schritt 11. **`forced_align` bleibt torch** —
nur der Modell-Forward wandert zu OpenVINO.

In [ ]:
from scipy.signal import butter, sosfiltfilt

# Signalkette vor dem ASR (alle Schritte sind linear, phasentreu oder nicht-neuronal
# -> beruehren die spektrale Signatur arabischer Gutturale/Emphatika/Frikative nicht):
#   decode  -> HP80Hz  -> RMS-Norm -> gentle_trim (VAD) -> Kontext-Pad
_HPF_SOS = butter(2, 80.0, btype="highpass", fs=SR, output="sos")

def decode_audio(raw: bytes) -> np.ndarray:
    """Beliebiges Audioformat -> 16 kHz mono float32."""
    seg = AudioSegment.from_file(io.BytesIO(raw))
    seg = seg.set_frame_rate(SR).set_channels(1).set_sample_width(2)
    return np.asarray(seg.get_array_of_samples(), dtype=np.float32) / 32768.0

def _highpass(audio: np.ndarray) -> np.ndarray:
    """Nullphasiger Butterworth @ 80 Hz: entfernt DC-Offset, Handling-Rumpeln,
    Netzbrummen (50/60 Hz). Liegt unterhalb jeder Sprachformantenergie."""
    return sosfiltfilt(_HPF_SOS, audio).astype(np.float32)

def _normalize_level(audio: np.ndarray, target_dbfs: float = -20.0) -> np.ndarray:
    """RMS-Normalisierung auf konsistentes Pegel -> Silero-VAD-Schwelle wird reproduzierbar,
    und wav2vec2s eingebautes do_normalize=True bekommt ein saubereres Zero-Mean/Unit-Var-Ziel."""
    rms = float(np.sqrt(np.mean(audio ** 2)))
    if rms < 1e-6:
        return audio
    gain = 10.0 ** ((target_dbfs - 20.0 * np.log10(rms)) / 20.0)
    out  = audio * gain
    peak = float(np.max(np.abs(out)))
    if peak > 0.99:
        out = out / peak * 0.99
    return out.astype(np.float32)

def gentle_trim(audio: np.ndarray, pad_ms: int = 120) -> np.ndarray:
    """Nur führende/nachlaufende lange Stille entfernen. Zwischenpausen bleiben."""
    segs = get_speech_timestamps(torch.from_numpy(audio), vad_model,
                                 sampling_rate=SR, threshold=0.35)
    if not segs:
        return audio
    pad = int(pad_ms * SR / 1000)
    start = max(0, segs[0]["start"] - pad)
    end   = min(len(audio), segs[-1]["end"] + pad)
    return audio[start:end]

def _pad_context(audio: np.ndarray, ms: int = 250) -> np.ndarray:
    """Wav2vec2-Transformer sieht pro Frame ein bidirektionales Kontextfenster (~200 ms).
    Kurze Woerter (2-3 Buchstaben) verlieren sonst am Anfang/Ende Kontextframes und werden
    systematisch schlechter erkannt. Silence-Padding kostet keine Latenz und keine Genauigkeit."""
    pad = np.zeros(int(ms * SR / 1000), dtype=np.float32)
    return np.concatenate([pad, audio, pad])

def preprocess(raw: bytes) -> np.ndarray:
    audio = decode_audio(raw)
    audio = _highpass(audio)
    audio = _normalize_level(audio)
    audio = gentle_trim(audio)
    audio = _pad_context(audio)
    return audio

In [ ]:
# Nur klassisches Tashkeel entfernen. Hamza-Formen (أ إ آ ؤ ئ) bleiben als eigene Buchstaben erhalten.
_TASHKEEL = set("ًٌٍَُِّْٰ")

def strip_diacritics(text: str) -> str:
    nfd = unicodedata.normalize("NFD", text)
    return unicodedata.normalize("NFC", "".join(c for c in nfd if c not in _TASHKEEL))

# Positionsabhaengige Aequivalenzen (Anfang/Ende) fuer Posterior-Bewertung.
_START_EQUIV = {ch: "اأإآ" for ch in "اأإآ"}
_END_EQUIV   = {"ة": "ةه", "ه": "هة",
                "ى": "ىيا", "ي": "يى"}

# Linguistisch belegte Verwechslungen fuer den LLR-Test.
# Quellen: Al-Ani (1970) "Arabic Phonology"; Newman (2013);
# Standard-DaF/L2-Arabisch-Fehlerkataloge; Kinder-L1-Erwerbsstudien.
_CONFUSABLES: Dict[str, str] = {
    "ت": "طثد",
    "ث": "تسذف",
    "ح": "هخع",
    "خ": "حغك",
    "د": "تضذ",
    "ذ": "دزثظ",
    "ر": "لغ",
    "ز": "ذسظ",
    "س": "صثزش",
    "ش": "سج",
    "ص": "سض",
    "ض": "دظص",
    "ط": "تضد",
    "ظ": "زذض",
    "ع": "ءأاه",
    "غ": "خقر",
    "ق": "كغخ",
    "ك": "قخج",
    "ل": "ر",
    "ه": "حة",
    "ء": "ع",
    "ج": "شك",
}

def _equiv_ids(ch: str, pos: int, total: int) -> List[int]:
    if pos == 0 and ch in _START_EQUIV:
        alts = _START_EQUIV[ch]
    elif pos == total - 1 and ch in _END_EQUIV:
        alts = _END_EQUIV[ch]
    else:
        alts = ch
    ids = [ASR_VOCAB[c] for c in alts if c in ASR_VOCAB]
    return ids or [ASR_VOCAB[ch]]

def _confuse_ids(ch: str) -> List[int]:
    alts = _CONFUSABLES.get(ch, "")
    return [ASR_VOCAB[c] for c in alts if c in ASR_VOCAB]

# Umkehr-Map: Token-ID -> Buchstabe, fuer error_hint.
_ID_TO_CHAR = {tid: c for c, tid in ASR_VOCAB.items()}

def encode_target(word: str) -> List[int]:
    ids: List[int] = []
    for ch in word:
        tid = ASR_VOCAB.get(ch)
        if tid is None:
            raise ValueError(f"Zeichen {ch!r} nicht im ASR-Vokabular.")
        ids.append(tid)
    return ids

@torch.inference_mode()
def run_asr(audio: np.ndarray):
    inputs = asr_processor(audio, sampling_rate=SR, return_tensors="pt", padding=True)
    input_values = inputs.input_values.to(device=device, dtype=DTYPE)
    logits = asr_model(input_values).logits
    # log_softmax stabil in fp32, torchaudio.forced_align verlangt float32 CPU.
    log_probs = torch.log_softmax(logits.float(), dim=-1).cpu()
    transcription = asr_processor.batch_decode(log_probs.argmax(dim=-1))[0]
    return log_probs, transcription

def _runs_of_non_blank(tokens: List[int]) -> List[List[int]]:
    runs: List[List[int]] = []
    current: List[int] = []
    last: int = -1
    for t, tok in enumerate(tokens):
        if tok == ASR_BLANK_ID:
            if current: runs.append(current); current = []
            last = -1
        elif tok != last:
            if current: runs.append(current)
            current = [t]; last = tok
        else:
            current.append(t)
    if current: runs.append(current)
    return runs

# Kalibrierungskonstante: LLR=0 -> 50, LLR=+1 -> ~88, LLR=-1 -> ~12.
_LLR_K = 2.0

def _sigmoid(x: float) -> float:
    return 1.0 / (1.0 + float(np.exp(-x)))

def gop_score(log_probs: torch.Tensor, target_word: str) -> List[Dict[str, Any]]:
    target_ids = encode_target(target_word)
    if not target_ids:
        return []
    if log_probs.shape[1] < len(target_ids):
        raise ValueError("Aufnahme zu kurz für dieses Wort.")
    targets = torch.tensor([target_ids], dtype=torch.int32)
    aligned, _ = AF.forced_align(log_probs, targets, blank=ASR_BLANK_ID)
    runs = _runs_of_non_blank(aligned[0].tolist())
    total_len = len(target_word)
    results: List[Dict[str, Any]] = []
    for i, ch in enumerate(target_word):
        if i >= len(runs):
            results.append({"label": ch, "score": 0.0, "confidence": 0.0,
                            "llr": -5.0, "error_hint": None})
            continue

        frames   = runs[i]
        lp_frame = log_probs[0, frames]  # [F, V]

        # 1) Posterior-Score (klassisches GOP, positionsbewusst).
        equiv_ids  = _equiv_ids(ch, i, total_len)
        target_lp  = lp_frame[:, equiv_ids].max(dim=-1).values.mean().item()
        post_score = float(np.clip((target_lp + 3.0) / 3.0 * 100, 0, 100))
        conf       = float(np.exp(target_lp))

        # 2) LLR gegen dokumentierte Verwechslungen (Anti-Modell).
        confuse_ids = _confuse_ids(ch)
        if confuse_ids:
            per_frame_conf = lp_frame[:, confuse_ids]
            best_conf_lp   = per_frame_conf.max(dim=-1).values.mean().item()
            llr            = target_lp - best_conf_lp
            llr_score      = _sigmoid(_LLR_K * llr) * 100.0
            # Nur melden wenn Verwechslung staerker als Ziel.
            if llr < 0:
                best_col   = int(per_frame_conf.mean(dim=0).argmax().item())
                hint_id    = confuse_ids[best_col]
                error_hint = _ID_TO_CHAR.get(hint_id)
            else:
                error_hint = None
        else:
            llr, llr_score, error_hint = 5.0, 100.0, None

        # 3) Kombination: 40 % Posterior + 60 % LLR (LLR ist informativer).
        final = 0.4 * post_score + 0.6 * llr_score
        results.append({
            "label": ch,
            "score": float(np.clip(final, 0, 100)),
            "confidence": conf,
            "llr": float(llr),
            "error_hint": error_hint,
        })
    return results

## Schritt 13 — FastAPI-App

Wörtlich aus `wav2vec2_arabic_pronunciation.ipynb` (Zelle 6): `/assess` (HTTP),
`/stream` (WebSocket, Wort- und Ayah-Modus), `/health`, `/logs`. Auth per
`API_TOKEN` — aus dem Colab-Secret `API_TOKEN`, sonst automatisch generiert und
in Schritt 14 ausgegeben.

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form, HTTPException, WebSocket, WebSocketDisconnect, Depends
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional, Iterator
import asyncio, json
from collections import deque
from datetime import datetime, timezone

MAX_AUDIO_BYTES     = 3 * 1024 * 1024
MAX_AYAH_AUDIO_BYTES = 8 * 1024 * 1024
MIN_SAMPLES     = int(0.15 * SR)

# Cloudflared Quick Tunnel killt WS-Verbindungen nach ~60-100s Inaktivitaet.
# Wir senden alle 20s einen App-Level-Ping, damit die Verbindung fuer die
# ganze Kind-Session offen bleibt (sonst Reconnect-Cost pro Wort ~500ms).
WS_KEEPALIVE_SEC = 20

# Zwischen Wort-Frames im Ayah-Stream: minimaler Delay, damit die UI
# das progressive Einfaerben visuell wahrnimmt statt "alles auf einmal".
WORD_STREAM_DELAY_SEC = 0.035

# ----- Timing-Log (In-Memory Ring + Datei, ueber GET /logs + !tail abrufbar) -----
_LOG_BUF: deque = deque(maxlen=200)
_LOG_FILE = "/content/backend.log"
try:
    # datei zuruecksetzen bei jedem Cell-Rerun
    open(_LOG_FILE, "w").close()
except Exception:
    _LOG_FILE = "/tmp/backend.log"
    try: open(_LOG_FILE, "w").close()
    except Exception: pass

def _log(event: str, **kv):
    """Struktur-Log: Colab-Cell + In-Memory-Ring + Datei /content/backend.log.
    Ansicht:  !tail -n 40 /content/backend.log   oder   <BACKEND_URL>/logs"""
    entry = {"ts": datetime.now(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z"),
             "event": event, **kv}
    _LOG_BUF.append(entry)
    kv_str = " ".join(f"{k}={v}" for k, v in kv.items())
    line = f"[{entry['ts']}] {event}  {kv_str}"
    print(line, flush=True)
    try:
        with open(_LOG_FILE, "a") as fh:
            fh.write(line + "\n"); fh.flush()
    except Exception:
        pass

class Unit(BaseModel):
    label: str
    score: float
    confidence: float
    llr: Optional[float] = None
    error_hint: Optional[str] = None

class AssessResponse(BaseModel):
    target: str
    transcription: str
    units: List[Unit]
    total: float
    duration_ms: int

app = FastAPI(title="Arabic Pronunciation API", version="2.0.0-colab")
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["*"], allow_headers=["*"])

# --- Auth ---------------------------------------------------------------
# API_TOKEN aus Colab-Secret oder auto-generiert. Muss auch im App-Setting
# (Settings -> "Auth-Token") gesetzt sein, sonst 401.
import secrets as _secrets
try:
    from google.colab import userdata as _ud
    API_TOKEN = _ud.get("API_TOKEN") or None
except Exception:
    API_TOKEN = None
if not API_TOKEN:
    API_TOKEN = os.environ.get("API_TOKEN") or _secrets.token_urlsafe(24)

from fastapi import Header, Query, status
from fastapi.responses import JSONResponse

def _require_token(header_token: Optional[str], query_token: Optional[str]) -> None:
    """Konstantzeit-Vergleich gegen API_TOKEN. Raises 401 bei Mismatch."""
    provided = header_token or query_token or ""
    if not provided or not _secrets.compare_digest(provided, API_TOKEN):
        raise HTTPException(status.HTTP_401_UNAUTHORIZED, "Ungueltiger oder fehlender API-Token.")

def _auth_dep(
    x_api_token: Optional[str] = Header(default=None, alias="X-API-Token"),
    token: Optional[str] = Query(default=None),
) -> None:
    _require_token(x_api_token, token)

# Middleware: jeder HTTP-Request wird geloggt (auch /health). Damit sieht man
# im /logs sofort, ob der Client ueberhaupt am Backend ankommt.
@app.middleware("http")
async def _request_logger(request, call_next):
    t0 = time.perf_counter()
    resp = await call_next(request)
    # /logs selbst NICHT loggen, sonst rekursive Flut beim Auto-Refresh.
    if not request.url.path.startswith("/logs"):
        _log("http",
             method=request.method,
             path=request.url.path,
             status=resp.status_code,
             ms=int((time.perf_counter() - t0) * 1000),
             client=request.client.host if request.client else "?")
    return resp

def _score_word(raw: bytes, target: str) -> Dict[str, Any]:
    """Synchrone Bewertungs-Pipeline. Wird sowohl vom HTTP- als auch vom WS-Endpoint aufgerufen,
    damit die Bewertungsqualitaet identisch bleibt."""
    if len(raw) > MAX_AUDIO_BYTES:
        raise HTTPException(413, f"Audio > {MAX_AUDIO_BYTES // 1024} KB.")
    if not raw:
        raise HTTPException(400, "Leere Audiodatei.")
    try:
        wav = preprocess(raw)
    except Exception as e:
        raise HTTPException(400, f"Audio ungültig: {e}")
    if wav.size < MIN_SAMPLES:
        raise HTTPException(400, "Aufnahme zu kurz.")
    target_clean = strip_diacritics(target)
    log_probs, transcription = run_asr(wav)
    units = gop_score(log_probs, target_clean)
    total = float(np.mean([u["score"] for u in units])) if units else 0.0
    return {
        "target": target_clean,
        "transcription": transcription,
        "units": units,
        "total": total,
    }

# ---------------------------------------------------------------------------
# Ayah-Modus: eine ganze Ayah (mehrere Woerter) in EINEM ASR-Forward-Pass
# scoren und pro Wort progressiv an den Client streamen.
#
# Design:
#   1. Alle Woerter der Ayah werden zu EINEM Buchstaben-Ziel konkateniert.
#      forced_align liefert damit ein globales, konsistentes Alignment ueber
#      die gesamte Rezitation - genauer als N unabhaengige Wort-Alignments,
#      weil Uebergaenge zwischen Woertern (Waslah, Idghaam) mitmodelliert werden.
#   2. Aus den Buchstaben-Runs werden per Wort-Grenzen-Map wieder Wort-Scores
#      aggregiert (Mittelwert der Buchstaben-Scores + Mindestwert-Penalty
#      falls einzelne Buchstaben stark abfallen).
#   3. Der WS-Handler yielded die Wort-Ergebnisse einzeln mit minimalem Delay,
#      damit der Client die Einfaerbung wie eine Live-Auswertung rendert.
#
# Woerter mit Tashkeel werden per strip_diacritics gecleant; Hamza-Formen
# bleiben erhalten (siehe _TASHKEEL). Wenn die Aufnahme zu kurz ist, um alle
# Buchstaben zu enthalten, bekommen fehlende Runs Score 0 - der Client kann
# das als "abgeschnitten" darstellen.
# ---------------------------------------------------------------------------

def _score_ayah_streamed(raw: bytes, ayah_text: str) -> Iterator[Dict[str, Any]]:
    if len(raw) > MAX_AYAH_AUDIO_BYTES:
        raise HTTPException(413, f"Audio > {MAX_AYAH_AUDIO_BYTES // 1024} KB.")
    if not raw:
        raise HTTPException(400, "Leere Audiodatei.")

    # 1) Ayah in Woerter zerlegen (Whitespace-basiert). Leere Tokens verwerfen.
    raw_words = [w for w in ayah_text.split() if w.strip()]
    if not raw_words:
        raise HTTPException(400, "Ayah-Text leer.")

    words_clean: List[str] = []
    for w in raw_words:
        cw = strip_diacritics(w)
        if cw:
            words_clean.append(cw)
    if not words_clean:
        raise HTTPException(400, "Ayah-Text enthaelt keine bewertbaren Zeichen.")

    # 2) Kompletter Buchstaben-Stream + Wort-Grenzen-Map [start, end).
    all_chars: List[str] = []
    word_spans: List[tuple] = []
    for w in words_clean:
        s = len(all_chars)
        all_chars.extend(list(w))
        word_spans.append((s, len(all_chars)))

    # 3) Ziel-IDs; unbekannte Zeichen -> Fehler mit klarer Meldung.
    target_ids: List[int] = []
    for ch in all_chars:
        tid = ASR_VOCAB.get(ch)
        if tid is None:
            raise HTTPException(400, f"Zeichen {ch!r} nicht im ASR-Vokabular.")
        target_ids.append(tid)

    # 4) Preprocess + ASR (ein Forward-Pass fuer die ganze Ayah).
    t_pre = time.perf_counter()
    try:
        wav = preprocess(raw)
    except Exception as e:
        raise HTTPException(400, f"Audio ungueltig: {e}")
    if wav.size < MIN_SAMPLES:
        raise HTTPException(400, "Aufnahme zu kurz.")
    dt_pre = int((time.perf_counter() - t_pre) * 1000)

    t_asr = time.perf_counter()
    log_probs, transcription = run_asr(wav)
    dt_asr = int((time.perf_counter() - t_asr) * 1000)

    if log_probs.shape[1] < len(target_ids):
        raise HTTPException(400, "Aufnahme zu kurz fuer diese Ayah.")

    t_align = time.perf_counter()
    targets = torch.tensor([target_ids], dtype=torch.int32)
    aligned, _ = AF.forced_align(log_probs, targets, blank=ASR_BLANK_ID)
    runs = _runs_of_non_blank(aligned[0].tolist())
    dt_align = int((time.perf_counter() - t_align) * 1000)

    # 5) Start-Frame: gibt dem Client die Wort-Anzahl + Transkription zur Anzeige.
    yield {
        "kind": "start",
        "words_count": len(words_clean),
        "transcription": transcription,
    }

    t_score = time.perf_counter()
    per_word_scores: List[float] = []
    for wi, (start_c, end_c) in enumerate(word_spans):
        word = words_clean[wi]
        word_len = end_c - start_c
        char_units: List[Dict[str, Any]] = []

        for local_i, char_global_i in enumerate(range(start_c, end_c)):
            ch = all_chars[char_global_i]
            if char_global_i >= len(runs):
                char_units.append({
                    "label": ch, "score": 0.0, "confidence": 0.0,
                    "llr": -5.0, "error_hint": None,
                })
                continue

            frames = runs[char_global_i]
            lp_frame = log_probs[0, frames]

            # Positionsbewusstsein: Anfang/Ende gilt PRO WORT, nicht pro Ayah.
            equiv_ids = _equiv_ids(ch, local_i, word_len)
            target_lp = lp_frame[:, equiv_ids].max(dim=-1).values.mean().item()
            post_score = float(np.clip((target_lp + 3.0) / 3.0 * 100, 0, 100))
            conf = float(np.exp(target_lp))

            confuse_ids = _confuse_ids(ch)
            if confuse_ids:
                per_frame_conf = lp_frame[:, confuse_ids]
                best_conf_lp = per_frame_conf.max(dim=-1).values.mean().item()
                llr = target_lp - best_conf_lp
                llr_score = _sigmoid(_LLR_K * llr) * 100.0
                if llr < 0:
                    best_col = int(per_frame_conf.mean(dim=0).argmax().item())
                    hint_id = confuse_ids[best_col]
                    error_hint = _ID_TO_CHAR.get(hint_id)
                else:
                    error_hint = None
            else:
                llr, llr_score, error_hint = 5.0, 100.0, None

            final = 0.4 * post_score + 0.6 * llr_score
            char_units.append({
                "label": ch,
                "score": float(np.clip(final, 0, 100)),
                "confidence": conf,
                "llr": float(llr),
                "error_hint": error_hint,
            })

        # Wort-Score: Mittelwert MIT Min-Penalty. Ein katastrophaler Buchstabe
        # zieht das Ergebnis staerker runter als reines Averaging, aber nicht
        # so hart, dass ein sonst gutes Wort komplett rot wird.
        char_scores = [u["score"] for u in char_units]
        mean_s = float(np.mean(char_scores)) if char_scores else 0.0
        min_s  = float(np.min(char_scores))  if char_scores else 0.0
        word_score = float(np.clip(0.75 * mean_s + 0.25 * min_s, 0, 100))
        per_word_scores.append(word_score)

        yield {
            "kind": "word",
            "word_idx": wi,
            "target": word,
            "score": word_score,
            "units": char_units,
        }

    total = float(np.mean(per_word_scores)) if per_word_scores else 0.0
    dt_score = int((time.perf_counter() - t_score) * 1000)
    yield {
        "kind": "done",
        "total": total,
        "words_count": len(words_clean),
        "timings": {
            "audio_bytes": len(raw),
            "audio_samples": int(wav.size),
            "audio_ms": int(wav.size * 1000 / SR),
            "preprocess_ms": dt_pre,
            "asr_ms": dt_asr,
            "align_ms": dt_align,
            "score_ms": dt_score,
        },
    }

@app.get("/health")
def health():
    return {"status": "ok", "device": str(device),
            "asr_model": ASR_MODEL_ID, "vad": "Silero VAD 5",
            "fp16": USE_FP16,
            "endpoints": ["/assess (HTTP)", "/stream (WebSocket)", "/logs"]}

@app.get("/logs", dependencies=[Depends(_auth_dep)])
def get_logs(n: int = 50, fmt: str = "html"):
    """Letzte N Log-Eintraege. Aus dem Handy-Browser:  <BACKEND_URL>/logs?n=40
    HTML mit Auto-Refresh (Standard) oder ?fmt=json fuer maschinell."""
    n = max(1, min(int(n), _LOG_BUF.maxlen or 200))
    entries = list(_LOG_BUF)[-n:][::-1]  # neueste oben
    if fmt == "json":
        return {"count": len(entries), "entries": entries}
    from fastapi.responses import HTMLResponse
    if not entries:
        rows = "<tr><td colspan='2' style='color:#94a3b8'>Noch keine Requests aufgezeichnet.</td></tr>"
    else:
        keys = ["ts", "event"] + sorted({k for e in entries for k in e if k not in ("ts", "event")})
        head = "".join(f"<th>{k}</th>" for k in keys)
        body_rows = []
        for e in entries:
            cells = "".join(f"<td>{e.get(k, '')}</td>" for k in keys)
            body_rows.append(f"<tr>{cells}</tr>")
        rows = f"<tr>{head}</tr>" + "".join(body_rows)
    html = f"""<!doctype html><html><head><meta charset='utf-8'>
<meta name='viewport' content='width=device-width,initial-scale=1'>
<meta http-equiv='refresh' content='2'>
<title>Backend-Logs</title>
<style>
body{{font-family:-apple-system,Segoe UI,Roboto,sans-serif;margin:0;padding:12px;background:#0f172a;color:#e2e8f0}}
h1{{font-size:15px;margin:0 0 8px}}
table{{width:100%;border-collapse:collapse;font-size:11px;font-family:ui-monospace,Menlo,Consolas,monospace}}
th{{text-align:left;padding:6px 8px;background:#1e293b;color:#93c5fd;position:sticky;top:0}}
td{{padding:5px 8px;border-top:1px solid #1e293b;color:#e2e8f0;white-space:nowrap}}
tr:nth-child(even) td{{background:#0b1220}}
.small{{color:#64748b;font-size:11px}}
</style></head><body>
<h1>Backend-Logs <span class='small'>· auto-refresh 2s · n={n}</span></h1>
<table>{rows}</table>
</body></html>"""
    return HTMLResponse(content=html)

@app.post("/assess", response_model=AssessResponse, dependencies=[Depends(_auth_dep)])
def assess(audio: UploadFile = File(...), target: str = Form(...)):
    target = target.strip()
    if not target:
        raise HTTPException(400, "Zielwort fehlt.")
    t0 = time.perf_counter()
    raw = audio.file.read(MAX_AUDIO_BYTES + 1)
    try:
        result = _score_word(raw, target)
    except ValueError as e:
        raise HTTPException(400, str(e))
    result["duration_ms"] = int((time.perf_counter() - t0) * 1000)
    return AssessResponse(units=[Unit(**u) for u in result["units"]], **{
        k: v for k, v in result.items() if k != "units"
    })

@app.websocket("/stream")
async def stream_ws(ws: WebSocket):
    # Auth via ?token=... (mobile-Client sendet den Token so).
    _q_token = ws.query_params.get("token") or ws.headers.get("x-api-token")
    if not _q_token or not _secrets.compare_digest(_q_token, API_TOKEN):
        await ws.close(code=1008, reason="invalid token")
        _log("ws_reject", reason="bad_token", client=ws.client.host if ws.client else "?")
        return
    """Persistente Session pro Kind. Zwei Modi ueber dasselbe WS.

    A) Einzelwort-Modus (bestehend, Play-Modus):
       1. Client -> Server: {"target": "kitab"} (Text-Frame)
       2. Client -> Server: Binaer-Frame mit Audio
       3. Server -> Client: AssessResponse-JSON

    B) Ayah-Modus (Quran-Reading, progressives Per-Wort-Streaming):
       1. Client -> Server: {"mode": "ayah", "ayah": "bism allah..."}
       2. Client -> Server: Binaer-Frame mit Audio
       3. Server -> Client (Serie):
            {"kind":"start","words_count":N,"transcription":"..."}
            {"kind":"word","word_idx":0,"target":"...","score":..,"units":[..]}
            ...
            {"kind":"done","total":..,"duration_ms":..}

    Zusaetzlich sendet der Server alle WS_KEEPALIVE_SEC Sekunden
    {"ping": true}, damit Cloudflared die Verbindung nicht als idle killt.
    """
    await ws.accept()
    _log("ws_open", client=ws.client.host if ws.client else "?")

    async def keepalive():
        try:
            while True:
                await asyncio.sleep(WS_KEEPALIVE_SEC)
                await ws.send_json({"ping": True})
        except Exception:
            return

    ka_task = asyncio.create_task(keepalive())

    try:
        while True:
            ctrl = json.loads(await ws.receive_text())
            mode = str(ctrl.get("mode", "word")).lower()

            if mode == "ayah":
                ayah = str(ctrl.get("ayah", "")).strip()
                if not ayah:
                    await ws.send_json({"error": "Ayah-Text fehlt."})
                    continue
                t_ctrl = time.perf_counter()
                msg = await ws.receive()
                if "bytes" not in msg or msg["bytes"] is None:
                    await ws.send_json({"error": "Erwartete Binaerdaten (Audio)."})
                    continue
                raw: bytes = msg["bytes"]
                dt_bytes_ms = int((time.perf_counter() - t_ctrl) * 1000)
                t0 = time.perf_counter()
                try:
                    frames = await asyncio.to_thread(
                        lambda: list(_score_ayah_streamed(raw, ayah))
                    )
                except HTTPException as e:
                    _log("ayah_err", detail=e.detail, bytes=len(raw))
                    await ws.send_json({"error": e.detail}); continue
                except ValueError as e:
                    _log("ayah_err", detail=str(e), bytes=len(raw))
                    await ws.send_json({"error": str(e)}); continue
                except Exception as e:
                    _log("ayah_err", detail=str(e), bytes=len(raw))
                    await ws.send_json({"error": f"Serverfehler: {e}"}); continue

                dt_compute = int((time.perf_counter() - t0) * 1000)
                t_stream = time.perf_counter()
                for f in frames:
                    if f.get("kind") == "done":
                        f["duration_ms"] = int((time.perf_counter() - t0) * 1000)
                        f.setdefault("timings", {})["bytes_recv_ms"] = dt_bytes_ms
                    await ws.send_json(f)
                    if f.get("kind") == "word":
                        await asyncio.sleep(WORD_STREAM_DELAY_SEC)
                dt_stream = int((time.perf_counter() - t_stream) * 1000)

                # Zusammenfassung fuer /logs und Colab-Cell.
                done = next((f for f in frames if f.get("kind") == "done"), {})
                t = done.get("timings", {})
                _log("ayah",
                     words=done.get("words_count"),
                     total=round(done.get("total", 0), 1),
                     bytes=len(raw),
                     audio_ms=t.get("audio_ms"),
                     recv=dt_bytes_ms,
                     pre=t.get("preprocess_ms"),
                     asr=t.get("asr_ms"),
                     align=t.get("align_ms"),
                     score=t.get("score_ms"),
                     stream=dt_stream,
                     compute=dt_compute)
                continue

            # --- Einzelwort-Modus (backwards compatible) ---
            target = str(ctrl.get("target", "")).strip()
            if not target:
                await ws.send_json({"error": "Zielwort fehlt."})
                continue
            msg = await ws.receive()
            if "bytes" not in msg or msg["bytes"] is None:
                await ws.send_json({"error": "Erwartete Binaerdaten (Audio)."})
                continue
            raw: bytes = msg["bytes"]
            t0 = time.perf_counter()
            try:
                result = await asyncio.to_thread(_score_word, raw, target)
            except HTTPException as e:
                await ws.send_json({"error": e.detail})
                continue
            except ValueError as e:
                await ws.send_json({"error": str(e)})
                continue
            except Exception as e:
                await ws.send_json({"error": f"Serverfehler: {e}"})
                continue
            result["duration_ms"] = int((time.perf_counter() - t0) * 1000)
            await ws.send_json(result)
    except WebSocketDisconnect:
        _log("ws_close", reason="disconnect")
        return
    except Exception as e:
        _log("ws_close", reason=f"exception:{e}")
        try: await ws.send_json({"error": f"Serverfehler: {e}"})
        except Exception: pass
    finally:
        ka_task.cancel()

print("✅ API definiert:  GET /health   POST /assess   WS /stream  (Wort+Ayah, Keep-Alive 20s)")

## Schritt 14 — Tailscale-Funnel und Server starten

Wörtlich aus `wav2vec2_arabic_pronunciation.ipynb` (Zelle 7).

**Voraussetzungen** (einmalig im Tailscale-Admin):

1. Colab-Secret **`TS_AUTHKEY`** setzen → https://login.tailscale.com/admin/settings/keys
2. **HTTPS aktivieren** → https://login.tailscale.com/admin/dns → *Enable HTTPS*
3. **Funnel-Attribut** in den ACLs → https://login.tailscale.com/admin/acls
   ```json
   "nodeAttrs": [ { "target": ["*"], "attr": ["funnel"] } ]
   ```
4. Optional Colab-Secret **`API_TOKEN`** — sonst wird einer generiert und unten
   ausgegeben.

Der erste `tailscale cert`-Aufruf kann bis zu 180 s dauern. Läuft die Zelle in
einen Timeout: einfach **nochmal ausführen**, der Rest ist idempotent.

Am Ende stehen `Backend-URL`, `WebSocket-URL` und `Token` in der Ausgabe — die
trägst du im Settings-Screen der App ein.

In [ ]:
import subprocess, json, os, shutil, threading, urllib.request, uvicorn, time as _t

PORT = 8000

# --- 0) Tailscale-Binaries installieren (idempotent). -------------------
if shutil.which("tailscale") is None:
    print("📦 Installiere Tailscale …")
    subprocess.run("curl -fsSL https://tailscale.com/install.sh | sh",
                   shell=True, check=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
else:
    print("✅ Tailscale bereits installiert")

# --- 1) tailscaled sauber (neu)starten ----------------------------------
subprocess.run("pkill -9 tailscaled || true", shell=True); _t.sleep(1)
os.makedirs("/var/run/tailscale", exist_ok=True)
os.makedirs("/var/lib/tailscale", exist_ok=True)
subprocess.Popen(
    ["tailscaled",
     "--tun=userspace-networking",
     "--socks5-server=localhost:1055",
     "--state=/var/lib/tailscale/tailscaled.state",
     "--socket=/var/run/tailscale/tailscaled.sock"],
    stdout=open("/tmp/tailscaled.log", "a"), stderr=subprocess.STDOUT,
)
for _ in range(20):
    if os.path.exists("/var/run/tailscale/tailscaled.sock"): break
    _t.sleep(0.3)
else:
    raise RuntimeError("❌ tailscaled-Socket nicht erreichbar. Log: /tmp/tailscaled.log")
print("✅ tailscaled läuft")

# --- 2) Auth-Key holen ---------------------------------------------------
_key = ""
try:
    from google.colab import userdata; _key = userdata.get("TS_AUTHKEY") or ""
except Exception:
    pass
_key = _key or os.environ.get("TS_AUTHKEY", "")
assert _key, "TS_AUTHKEY fehlt in Colab-Secret (https://login.tailscale.com/admin/settings/keys)"

# --- 3) tailscale up MIT TIMEOUT (kein Hang) -----------------------------
r = subprocess.run(
    ["tailscale", "up", f"--auth-key={_key}",
     "--hostname=colab-asr", "--accept-routes=false", "--timeout=30s"],
    capture_output=True, text=True, timeout=60,
)
print("up rc:", r.returncode, " stderr:", (r.stderr or "").strip()[:200])
if r.returncode != 0:
    raise RuntimeError(f"❌ 'tailscale up' fehlgeschlagen: {r.stderr.strip()}")

# --- 4) Auf echten Online-Status warten (Self.Online == True). -----------
# Hostname-Match allein reicht nicht: der Node erscheint sofort im Status,
# ist aber typisch 5-20s laenger "offline", bis er das erste Handshake
# mit dem Control-Plane und einem DERP-Relay abgeschlossen hat. 'funnel'
# haengt genau in diesem Fenster.
_self = {}
for _ in range(60):
    try:
        _st_j = json.loads(subprocess.check_output(
            ["tailscale", "status", "--json"], timeout=5))
        _self = _st_j.get("Self") or {}
        if _self.get("Online") is True:
            break
    except Exception:
        pass
    _t.sleep(1)
else:
    raise RuntimeError(
        "❌ Node ist nach 60s noch offline (Self.Online=False).\n"
        f"   Self: hostname={_self.get('HostName')} online={_self.get('Online')}\n"
        "   Meist reicht ein zweiter Zellen-Run; sonst tailscaled-Log pruefen:\n"
        "   !tail -n 40 /tmp/tailscaled.log"
    )
print(f"✅ Node online:  {_self.get('HostName')}  ({_self.get('DNSName','').rstrip('.')})")

# --- 5) FastAPI: starten falls noch nicht erreichbar ---------------------
def _fastapi_up() -> bool:
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2).read()
        return True
    except Exception:
        return False

if _fastapi_up():
    print("✅ FastAPI antwortet (alter Prozess)")
else:
    print("⚠️ FastAPI nicht erreichbar, starte neu …")
    threading.Thread(
        target=lambda: uvicorn.run(app, host="0.0.0.0", port=PORT,
                                   log_level="warning", access_log=False),
        daemon=True,
    ).start()
    for _ in range(30):
        if _fastapi_up(): break
        _t.sleep(0.5)
    else:
        raise RuntimeError("❌ FastAPI-Start fehlgeschlagen.")
    print(f"✅ FastAPI gestartet auf Port {PORT}")

# --- 6) Public DNSName ermitteln (fuer Cert + finale URL) ----------------
DNS_NAME   = (_self.get("DNSName") or "").rstrip(".")
if not DNS_NAME:
    raise RuntimeError("❌ Konnte DNSName nicht aus 'tailscale status --json' lesen.")
public_url = f"https://{DNS_NAME}"

# --- 7) Funnel-Aktivierung ----------------------------------------------
# 'tailscale funnel --bg <port>' blockt in Colab manchmal bis der Server
# den Share bestaetigt hat - Timeouts sind hier nicht zuverlaessig. Wir
# starten den Befehl als Popen und pollen parallel 'funnel status', damit
# wir sofort abbrechen koennen sobald der Share sichtbar ist.

def _funnel_status_blob() -> str:
    try:
        fs = subprocess.run(["tailscale", "funnel", "status"],
                            capture_output=True, text=True, timeout=5)
        return (fs.stdout or "") + (fs.stderr or "")
    except Exception:
        return ""

def _funnel_serving(port: int) -> bool:
    blob = _funnel_status_blob()
    return f"127.0.0.1:{port}" in blob or f":{port}" in blob

if _funnel_serving(PORT):
    print(f"✅ Funnel bereits aktiv fuer Port {PORT}")
else:
    # --- 7a) TLS-Zert PROAKTIV holen ------------------------------------
    # Der eigentliche Hang beim ersten 'funnel'-Aufruf ist das synchrone
    # ACME-Zertifikat-Ausstellen (kann 30-90s dauern). Wir triggern das
    # explizit mit grossem Timeout - danach ist 'funnel' unter 1s fertig.
    cert_path = f"/var/lib/tailscale/certs/{DNS_NAME}.crt"
    if not os.path.exists(cert_path):
        print(f"🔐 Stelle TLS-Zert fuer {DNS_NAME} aus (kann 30-90s dauern) …")
        try:
            cr = subprocess.run(
                ["tailscale", "cert", DNS_NAME],
                capture_output=True, text=True, timeout=180,
                cwd="/tmp",
            )
            if cr.returncode != 0:
                msg = (cr.stderr or cr.stdout or "").strip()
                low = msg.lower()
                if "https is not enabled" in low or "enablehttps" in low or "https must be enabled" in low:
                    raise RuntimeError(
                        "❌ HTTPS im Tailscale-Admin nicht aktiviert.\n"
                        "   https://login.tailscale.com/admin/dns  →  'Enable HTTPS'\n"
                        f"   stderr: {msg}"
                    )
                raise RuntimeError(f"❌ 'tailscale cert' fehlgeschlagen: {msg}")
            print("✅ TLS-Zert ausgestellt")
        except subprocess.TimeoutExpired:
            raise RuntimeError(
                "❌ 'tailscale cert' nach 180s nicht fertig. Zelle NOCHMAL ausfuehren -\n"
                "   das Zert wird meist im Hintergrund weiter ausgestellt."
            )
    else:
        print("✅ TLS-Zert bereits vorhanden")

    # --- 7b) Funnel als Popen + parallel pollen -------------------------
    print(f"📡 Aktiviere Funnel auf Port {PORT} …")
    _fp = subprocess.Popen(
        ["tailscale", "funnel", "--bg", str(PORT)],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
    )
    _deadline = _t.time() + 90
    _served = False
    while _t.time() < _deadline:
        if _funnel_serving(PORT):
            _served = True
            break
        if _fp.poll() is not None:
            break
        _t.sleep(1)

    # Prozess sauber einsammeln (nicht blockieren, falls er noch laeuft).
    _out, _err = "", ""
    if _fp.poll() is None:
        if _served:
            # Funnel steht - Kommando hat seinen Zweck erfuellt, killen.
            _fp.terminate()
            try: _out, _err = _fp.communicate(timeout=5)
            except Exception:
                _fp.kill()
                try: _out, _err = _fp.communicate(timeout=5)
                except Exception: pass
        else:
            _fp.kill()
            try: _out, _err = _fp.communicate(timeout=5)
            except Exception: pass
    else:
        try: _out, _err = _fp.communicate(timeout=5)
        except Exception: pass

    print("--- funnel ---")
    if _out: print("stdout:", _out.strip())
    if _err: print("stderr:", _err.strip())

    # Endgueltige Wahrheit: 'funnel status'. Ob 'funnel' rc!=0 oder
    # gekilled wurde ist egal, solange der Share aktiv ist.
    if not _funnel_serving(PORT):
        msg = (_err or _out or "").strip().lower()
        if "funnel" in msg and "attribute" in msg:
            raise RuntimeError(
                "❌ Funnel-ACL nicht gesetzt.\n"
                "   https://login.tailscale.com/admin/acls  ergaenze:\n"
                '     "nodeAttrs": [ { "target": ["*"], "attr": ["funnel"] } ]\n'
                f"   stderr: {(_err or _out).strip()}"
            )
        raise RuntimeError(
            "❌ Funnel wurde nach 90s nicht aktiv.\n"
            f"   funnel status:\n{_funnel_status_blob() or '(leer)'}\n"
            "   Zelle nochmal ausfuehren; alternativ manuell testen mit:\n"
            f"   !tailscale funnel --bg {PORT}"
        )
    print("✅ Funnel aktiv")

print("\n" + "=" * 68)
print(f"🌍 Backend-URL für die App:  {public_url}")
print(f"🔐 API-Token (in App eintragen):  {API_TOKEN}")
print("=" * 68)
print(f"Health-Check:  {public_url}/health")
print(f"WebSocket:     {public_url.replace('https://', 'wss://')}/stream?token={API_TOKEN}")
print(f"Live-Logs:     {public_url}/logs?token={API_TOKEN}")


## 🔎 Backend-Logs anschauen

Diese Zelle jederzeit **erneut ausführen**, um die letzten Backend-Timings zu sehen.  
Alternativ im Handy-Browser: `{PUBLIC_URL}/logs`  (HTML mit Auto-Refresh).


In [ ]:
# --- Backend-Logs Live (jederzeit erneut ausfuehren) ---
import subprocess
out = subprocess.run(["tail", "-n", "40", "/content/backend.log"], capture_output=True, text=True)
print(out.stdout or "(noch keine Logs)")
if 'public_url' in dir():
    print(f"\n🌍 Live im Browser: {public_url}/logs")


---

## Was beim Testen mit der App zu beobachten ist

* **`asr_ms` in den Timings** (`/logs` oder die Response) ist die Zahl, die zählt.
  Auf Colab-CPU (2 geteilte vCPU) wird sie deutlich über dem 300-ms-Budget liegen —
  das ist ein Artefakt der Colab-Hardware, nicht des Exports. Für belastbare Werte
  auf der Zielhardware messen.
* **Erste Anfrage je Audiolänge ist langsamer** (dynamische Shape → OpenVINO
  re-optimiert). Für stabile Latenz auf feste Buckets padden (siehe Schritt 6).
* **INT8 vs. FP32 vergleichen:** `VARIANT` in Schritt 11 umstellen, Schritt 11 und
  14 erneut ausführen, gleiche Aufnahmen sprechen und die `score`-Werte pro
  Buchstabe vergleichen. Weichen sie systematisch ab, ist das der GOP-Drift aus
  Schritt 7 — dann FP32 nehmen oder mit echten Kinderaufnahmen neu kalibrieren.

## Für die Übernahme ins Backend

* **Frame-Raster unverändert.** Der IR-Export ändert den Conv-Stack nicht -> die
  ~20 ms/Frame und `blank_id = config.pad_token_id` gelten weiter. In
  `backend/asr_app.py` muss nur `_run_asr()` ausgetauscht werden; `_gop_score()`,
  `_runs_of_non_blank()`, `forced_align` und die Tajweed-Schwellen bleiben gleich.
* **`forced_align` braucht weiter torch/torchaudio.** Nur der Modell-Forward
  wandert zu OpenVINO. Wer torch komplett loswerden will, muss zuerst
  `torchaudio.functional.forced_align` ersetzen.
* **Latenzbudget entscheidet sich über Shapes, nicht über INT8 allein.** Statische
  Shapes + `CACHE_DIR` sind der zuverlässigste Hebel; INT8 kommt oben drauf.
* **Vor der Quantisierung erst die billigen Gewinne holen:** der nie erreichte
  Silero-VAD-Load und die 250 ms Padding auf jeder Seite (= 25 zusätzliche Frames
  pro Anfrage) kosten auf CPU direkt Rechenzeit.
* **INT8 nur mit echten Kinderaufnahmen kalibrieren** (~64-128 Dateien nach
  `/content/calib_audio/`), und über `dGOP` abnehmen, nicht über das Transkript.